# Nova Geo Marts EDA

This notebook reads the geography-focused dbt marts from BigQuery into pandas DataFrames and performs lightweight EDA before any demand modeling work.

Primary tables:

- `mart_geo_market_opportunity`: market-level opportunity ranking
- `mart_geo_market_category_day_features`: market-category-day modeling feature table
- `mart_geo_weather_category_sensitivity`: market-category weather sensitivity summary

Power BI should ultimately read curated BigQuery/dbt tables. This notebook is for exploration and model preparation only.

## 1. Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import plotly.express as px
from google.cloud import bigquery

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

px.defaults.template = "plotly_white"
px.defaults.width = 1_050
px.defaults.height = 560

## 2. BigQuery Connector

This uses your active Google Application Default Credentials or the credentials available in the notebook environment. If authentication fails, run `gcloud auth application-default login` outside the notebook and retry.

In [2]:
PROJECT_ID = "nova-project-498911"
DATASET_ID = "dbt_doruk"

TABLES = {
    "market_opportunity": "mart_geo_market_opportunity",
    "market_category_day_features": "mart_geo_market_category_day_features",
    "weather_category_sensitivity": "mart_geo_weather_category_sensitivity",
}

client = bigquery.Client(project=PROJECT_ID)


def table_ref(table_name: str) -> str:
    return f"`{PROJECT_ID}.{DATASET_ID}.{table_name}`"


def read_bq(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()


dataset = client.get_dataset(f"{PROJECT_ID}.{DATASET_ID}")
print(f"Connected to {dataset.full_dataset_id}")

Connected to nova-project-498911:dbt_doruk


## 3. Read Geo Marts

In [3]:
market_opportunity = read_bq(
    f"""
    select *
    from {table_ref(TABLES["market_opportunity"])}
    order by opportunity_rank
    """
)

market_category_day_features = read_bq(
    f"""
    select *
    from {table_ref(TABLES["market_category_day_features"])}
    """
)

weather_category_sensitivity = read_bq(
    f"""
    select *
    from {table_ref(TABLES["weather_category_sensitivity"])}
    order by category, category_weather_sensitivity_rank
    """
)

dfs = {
    "market_opportunity": market_opportunity,
    "market_category_day_features": market_category_day_features,
    "weather_category_sensitivity": weather_category_sensitivity,
}

for name, df in dfs.items():
    print(f"{name:<32} {df.shape[0]:>8,} rows x {df.shape[1]:>3} columns")

/Users/doruk/dev/nova-analytics/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


market_opportunity                     16 rows x  76 columns
market_category_day_features       29,280 rows x  80 columns
weather_category_sensitivity           80 rows x  28 columns


In [4]:
for df in [market_opportunity, market_category_day_features, weather_category_sensitivity]:
    for col in df.columns:
        if col.endswith("_date") or col == "order_date":
            df[col] = pd.to_datetime(df[col])

market_category_day_features["year_month"] = market_category_day_features["order_date"].dt.to_period("M").astype(str)

## 4. Table Health Checks

In [5]:
health_rows = []
for name, df in dfs.items():
    health_rows.append(
        {
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum()),
            "total_null_cells": int(df.isna().sum().sum()),
        }
    )

table_health = pd.DataFrame(health_rows)
table_health

,table,rows,columns,duplicate_rows,total_null_cells
0,market_opportunity,16,76,0,0
1,market_category_day_features,29280,80,0,0
2,weather_category_sensitivity,80,28,0,0


In [6]:
grain_checks = pd.DataFrame(
    [
        {
            "table": "market_opportunity",
            "expected_grain": "market_id",
            "rows": len(market_opportunity),
            "distinct_grain": market_opportunity[["market_id"]].drop_duplicates().shape[0],
        },
        {
            "table": "market_category_day_features",
            "expected_grain": "market_id + category + order_date",
            "rows": len(market_category_day_features),
            "distinct_grain": market_category_day_features[["market_id", "category", "order_date"]].drop_duplicates().shape[0],
        },
        {
            "table": "weather_category_sensitivity",
            "expected_grain": "market_id + category",
            "rows": len(weather_category_sensitivity),
            "distinct_grain": weather_category_sensitivity[["market_id", "category"]].drop_duplicates().shape[0],
        },
    ]
)

grain_checks["grain_ok"] = grain_checks["rows"] == grain_checks["distinct_grain"]
grain_checks

,table,expected_grain,rows,distinct_grain,grain_ok
0,market_opportunity,market_id,16,16,True
1,market_category_day_features,market_id + category + order_date,29280,29280,True
2,weather_category_sensitivity,market_id + category,80,80,True


In [7]:
market_category_day_features[["order_date", "market_id", "category"]].agg(
    {
        "order_date": ["min", "max", "nunique"],
        "market_id": "nunique",
        "category": "nunique",
    }
)

,order_date,market_id,category
min,2024-01-01 00:00:00,NaN,NaN
max,2024-12-31 00:00:00,NaN,NaN
nunique,366,16.0000,5.0000


## 5. Market Opportunity Overview

In [20]:
market_cols = [
    "opportunity_rank",
    "market_name",
    "nova_region",
    "opportunity_segment",
    "opportunity_score",
    "current_performance_score",
    "macro_potential_score",
    "supply_score",
    "adoption_score",
    "total_transactions",
    "total_gmv_usd",
    "annual_active_users",
    "country_annual_active_users",
    "country_active_user_penetration",
    "country_urban_active_user_penetration",
    "transaction_growth_rate",
    "completion_rate",
]

market_opportunity[market_cols]

,opportunity_rank,market_name,nova_region,opportunity_segment,opportunity_score,current_performance_score,macro_potential_score,supply_score,adoption_score,total_transactions,total_gmv_usd,annual_active_users,country_annual_active_users,country_active_user_penetration,country_urban_active_user_penetration,transaction_growth_rate,completion_rate
0,1,Singapore,Southeast Asia,Growth candidate,70.2800,62.0926,71.5565,53.2761,100.0000,3352367,"161,072,839.4700",118547,118547,0.0196,0.0196,0.7105,0.9168
1,2,Jakarta,Southeast Asia,Maintain,54.0900,100.0000,23.0222,64.3810,2.2776,4711771,"226,445,740.3600",153184,153184,0.0005,0.0009,0.7088,0.9165
2,3,Dubai,EMEA,Maintain,50.2600,38.7550,66.4949,58.3359,42.0235,2524597,"121,398,439.0500",85413,85413,0.0078,0.0091,0.7117,0.9164
3,4,Mumbai,South Asia,Maintain,47.1500,87.2006,20.0000,57.8979,0.2391,4258852,"204,585,106.2200",135870,254685,0.0002,0.0005,0.7116,0.9164
4,5,Bangalore,South Asia,Maintain,46.9700,71.4204,20.0000,84.6437,0.2391,3694027,"177,517,395.7000",118815,254685,0.0002,0.0005,0.7224,0.9166
5,6,New York,North America,Maintain,46.5800,52.2259,59.5140,66.7530,0.3810,2997731,"143,979,341.1000",110138,110138,0.0003,0.0004,0.7179,0.9168
6,7,London,Europe,Monitor,44.0400,46.5340,51.4260,67.5970,6.8981,2795924,"134,338,396.6900",102155,102155,0.0015,0.0018,0.7094,0.9160
7,8,Manila,Southeast Asia,Monitor,42.9200,76.1940,15.1537,55.7988,6.5121,3862139,"185,462,470.2900",127918,127918,0.0011,0.0020,0.7138,0.9153
8,9,Bangkok,Southeast Asia,Monitor,39.1400,58.5744,38.0804,36.6726,8.9191,3229663,"155,250,080.6200",110298,110298,0.0015,0.0025,0.7096,0.9168
9,10,Tokyo,East Asia,Monitor,38.7500,47.2247,53.7388,41.0087,2.9332,2820571,"135,566,477.9900",102023,102023,0.0008,0.0009,0.7080,0.9161


In [9]:
fig = px.bar(
    market_opportunity.sort_values("opportunity_score"),
    x="opportunity_score",
    y="market_name",
    color="opportunity_segment",
    orientation="h",
    title="Market Opportunity Score by Market",
    labels={"market_name": "Market", "opportunity_score": "Opportunity score"},
)
fig.show()

In [10]:
fig = px.scatter_geo(
    market_opportunity,
    lat="latitude",
    lon="longitude",
    size="total_gmv_usd",
    color="opportunity_segment",
    hover_name="market_name",
    hover_data={
        "opportunity_score": ":.2f",
        "total_transactions": ":,",
        "total_gmv_usd": ":,.0f",
        "country_active_user_penetration": ":.3%",
        "transaction_growth_rate": ":.2%",
        "latitude": False,
        "longitude": False,
    },
    title="Market Opportunity Atlas",
)
fig.update_geos(projection_type="natural earth")
fig.show()

In [11]:
country_adoption = (
    market_opportunity[[
        "country_name",
        "country_iso3",
        "population_total",
        "country_annual_active_users",
        "country_active_user_penetration",
        "country_urban_active_user_penetration",
    ]]
    .drop_duplicates(subset=["country_iso3"])
    .sort_values("country_active_user_penetration", ascending=True)
)

fig = px.bar(
    country_adoption,
    x="country_active_user_penetration",
    y="country_name",
    orientation="h",
    color="country_urban_active_user_penetration",
    title="Country-Level Active User Penetration",
    labels={
        "country_active_user_penetration": "Active users / country population",
        "country_urban_active_user_penetration": "Active users / urban population",
        "country_name": "Country",
    },
    hover_data={
        "country_annual_active_users": ":,",
        "population_total": ":,",
        "country_active_user_penetration": ":.3%",
        "country_urban_active_user_penetration": ":.3%",
    },
)
fig.update_layout(xaxis_tickformat=".2%")
fig.show()

In [12]:
score_components = market_opportunity[
    [
        "market_name",
        "current_performance_score",
        "macro_potential_score",
        "supply_score",
        "adoption_score",
    ]
].melt(id_vars="market_name", var_name="score_component", value_name="score")

fig = px.bar(
    score_components,
    x="market_name",
    y="score",
    color="score_component",
    barmode="group",
    title="Opportunity Score Components",
    labels={"market_name": "Market", "score": "Score"},
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

## 6. Market-Category-Day Demand Shape

In [13]:
monthly_market = (
    market_category_day_features.groupby(["year_month", "market_name"], as_index=False)
    .agg(transactions=("transactions", "sum"), gmv_usd=("gmv_usd", "sum"), active_users=("active_users", "sum"))
)

fig = px.line(
    monthly_market,
    x="year_month",
    y="transactions",
    color="market_name",
    title="Monthly Transactions by Market",
    labels={"year_month": "Month", "transactions": "Transactions", "market_name": "Market"},
)
fig.show()

In [14]:
category_summary = (
    market_category_day_features.groupby("category", as_index=False)
    .agg(
        transactions=("transactions", "sum"),
        gmv_usd=("gmv_usd", "sum"),
        avg_daily_transactions=("transactions", "mean"),
        avg_transaction_amount_usd=("avg_transaction_amount_usd", "mean"),
        completion_rate=("completion_rate", "mean"),
        promo_share=("promo_share", "mean"),
    )
    .sort_values("transactions", ascending=False)
)

category_summary

,category,transactions,gmv_usd,avg_daily_transactions,avg_transaction_amount_usd,completion_rate,promo_share
2,Food Delivery,14761005,"397,435,777.2300","2,520.6634",26.9247,0.9434,0.2514
1,E-Commerce,11015252,"974,997,063.5400","1,881.0198",87.5591,0.8724,0.1667
4,Ride Hailing,10898345,"286,240,115.4500","1,861.0562",26.2640,0.8883,0.2514
3,Grocery,7287050,"488,284,173.0300","1,244.3733",67.0021,0.9184,0.0000
0,Digital Wallet,6038348,"255,760,195.9400","1,031.1387",42.3737,0.9784,0.0000


In [15]:
fig = px.bar(
    category_summary,
    x="category",
    y="transactions",
    color="category",
    title="Annual Transactions by Category",
    labels={"category": "Category", "transactions": "Transactions"},
)
fig.show()

In [16]:
market_category_mix = (
    market_category_day_features.groupby(["market_name", "category"], as_index=False)
    .agg(transactions=("transactions", "sum"))
)
market_category_mix["market_category_share"] = market_category_mix["transactions"] / market_category_mix.groupby("market_name")["transactions"].transform("sum")

fig = px.bar(
    market_category_mix,
    x="market_name",
    y="market_category_share",
    color="category",
    title="Category Mix by Market",
    labels={"market_name": "Market", "market_category_share": "Share of market transactions"},
)
fig.update_layout(xaxis_tickangle=-35, yaxis_tickformat=".0%")
fig.show()

## 7. Weather and Local Demand Checks

In [17]:
weather_market = (
    market_category_day_features.groupby("market_name", as_index=False)
    .agg(
        rain_day_share=("is_rain_day", "mean"),
        avg_temperature_c=("temperature_2m_mean_c", "mean"),
        avg_precipitation_mm=("precipitation_sum_mm", "mean"),
        avg_daily_transactions=("transactions", "mean"),
    )
    .sort_values("rain_day_share", ascending=False)
)

weather_market

,market_name,rain_day_share,avg_temperature_c,avg_precipitation_mm,avg_daily_transactions
13,Singapore,0.9754,26.7866,10.5298,"1,831.8945"
5,Jakarta,0.8579,27.3967,6.8311,"2,574.7383"
7,Manila,0.7896,28.0230,6.3082,"2,110.4585"
3,Ho Chi Minh City,0.7158,28.1260,5.8052,"1,702.0907"
1,Bangkok,0.6858,28.7918,4.4727,"1,764.8432"
6,London,0.6557,11.7238,2.4415,"1,527.8273"
14,Sydney,0.6175,17.8352,2.9183,623.4131
12,Sao Paulo,0.5656,20.3637,3.4691,"1,630.0279"
0,Bangalore,0.5601,24.0268,2.9533,"2,018.5940"
15,Tokyo,0.5383,16.6926,4.7030,"1,541.2956"


In [18]:
fig = px.scatter(
    weather_market,
    x="avg_temperature_c",
    y="rain_day_share",
    size="avg_daily_transactions",
    color="avg_precipitation_mm",
    hover_name="market_name",
    title="Weather Profile by Market",
    labels={
        "avg_temperature_c": "Average temperature, C",
        "rain_day_share": "Rain-day share",
        "avg_precipitation_mm": "Avg precipitation, mm",
    },
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [19]:
rain_lift = weather_category_sensitivity.copy()
rain_lift["rain_transaction_lift_pct"] = pd.to_numeric(rain_lift["rain_transaction_lift_pct"], errors="coerce")

fig = px.bar(
    rain_lift.sort_values("rain_transaction_lift_pct"),
    x="rain_transaction_lift_pct",
    y="market_name",
    color="rain_sensitivity_segment",
    facet_col="category",
    facet_col_wrap=2,
    title="Rain Transaction Lift by Market and Category",
    labels={"rain_transaction_lift_pct": "Rain transaction lift", "market_name": "Market"},
)
fig.update_xaxes(tickformat=".0%")
fig.show()

## 8. Notes and Modeling Candidates

Use this section to capture observations before moving to the dedicated demand-modeling notebook.

- Candidate target: `transactions`
- Secondary target: `gmv_usd`
- Candidate split: train through October 2024, validate on November-December 2024
- Candidate features: market context, calendar fields, promo share, service supply, platform mix, weather, macro indicators
- Candidate outputs to write back to BigQuery later: predictions, feature importance, model metrics, and market/category segment labels